In [0]:
%pip install databricks-feature-engineering

# Feature Engineering - Gold Layer (Produção)

Pipeline estável de feature engineering para detecção de fraude.

**Fonte:** `fraud_detection_dev.silver.fraud_data_clean`  
**Destino:** `fraud_detection_dev.gold.fraud_features` (Feature Store)

**Etapas:**
1. Split temporal (60/20/20)
2. Feature selection (qualidade, evidência, redundância)
3. Feature engineering (temporal, email, agregações)
4. Publicação no Unity Catalog Feature Store

**Uso:** Pipeline automatizado de retreino. Para experimentação, use `notebooks/02_FE_PRE_PROCE`.

In [0]:
from pyspark.sql import functions as F, Window
import sys

sys.path.insert(0, "/Workspace/Users/eduardolimaofficial410@gmail.com/databricks-fraud-detection/src")
from databricks_fraud_model.utils import load_config

cfg = load_config()
CATALOG = cfg["catalog"]
FEATURE_TABLE = cfg['feature_store']['table']

# batch_type: train (treino+test+val) ou prod
try:
    batch_type = dbutils.widgets.get("batch_type")
except Exception:
    batch_type = "train"

SOURCE_TABLE = f"{CATALOG}.silver.{batch_type}_data_clean"

In [0]:
dados = spark.sql(f"SELECT * FROM {SOURCE_TABLE}")

from databricks_fraud_model.features import FEATURES_DROP as features_drop, apply_feature_engineering_spark

if batch_type == "train":
    q60, q80 = dados.approxQuantile('TransactionDT', [0.60, 0.80], 0.01)
    train_df = dados.filter(F.col('TransactionDT') <= q60)
    test_df = dados.filter((F.col('TransactionDT') > q60) & (F.col('TransactionDT') <= q80))
    val_df = dados.filter(F.col('TransactionDT') > q80)

    train_s = train_df.drop(*[c for c in features_drop if c in train_df.columns])
    test_s = test_df.drop(*[c for c in features_drop if c in test_df.columns])
    val_s = val_df.drop(*[c for c in features_drop if c in val_df.columns])
    train_s = apply_feature_engineering_spark(train_s)
    test_s = apply_feature_engineering_spark(test_s)
    val_s = apply_feature_engineering_spark(val_s)

    from databricks.feature_engineering import FeatureEngineeringClient

    uc_parts = FEATURE_TABLE.split(".")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {uc_parts[0]}.{uc_parts[1]}")

    full_df = (
        train_s.withColumn("split", F.lit("train"))
        .unionByName(test_s.withColumn("split", F.lit("test")))
        .unionByName(val_s.withColumn("split", F.lit("val")))
        .filter(F.col("TransactionID").isNotNull())
        .withColumn("TransactionDT", F.from_unixtime(F.col("TransactionDT")).cast("timestamp"))
    )

    spark.sql(f"DROP TABLE IF EXISTS {FEATURE_TABLE}")

    fe = FeatureEngineeringClient()
    fe.create_table(
        name=FEATURE_TABLE,
        primary_keys=["TransactionID", "TransactionDT"],
        timeseries_column="TransactionDT",
        df=full_df,
        schema=full_df.schema,
        description="Feature table Gold para deteccao de fraude - features engenheiradas em formato RAW, split temporal",
    )

    spark.sql(
        f"COMMENT ON COLUMN {FEATURE_TABLE}.split IS "
        f"'Conjunto de dados: train, test ou val (split temporal)'"
    )
else:
    prod_s = dados.drop(*[c for c in features_drop if c in dados.columns])
    prod_s = apply_feature_engineering_spark(prod_s)

    GOLD_PRD = f"{CATALOG}.gold.fraud_features_prd"
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

    (
        prod_s.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(GOLD_PRD)
    )
    spark.sql(f"OPTIMIZE {GOLD_PRD} ZORDER BY (TransactionID)")